# Phase 2 - Part B: Generative AI Integration

## 1. Generative AI Model Setup

We use **LLaMA 3.3 70B** (Meta's open-source LLM) accessed through the **Groq API**.

**Why Groq + LLaMA?**
- Free API access (no credit card required)
- Open-source model aligns with academic transparency
- Fast inference via Groq's specialized hardware
- Listed as an option in the project handbook


In [8]:
# Import required libraries
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key from .env file
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# Initialize the Groq client
client = Groq(api_key=api_key)

# Model configuration
MODEL_NAME = "llama-3.3-70b-versatile"

# Quick connection test
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Reply with: connection OK"}],
    max_tokens=10
)

print("Model:", MODEL_NAME)
print("Response:", response.choices[0].message.content)

Model: llama-3.3-70b-versatile
Response: connection OK


## 2. Prompt Template Design

### Template 1: Simple Explanation

**Goal:** Explain the prediction to the patient in plain, friendly language.

**Target user:** Patients with no medical background.

**Design choice:** Short response (3-5 sentences), no medical jargon, reassuring tone.

In [2]:
# Template 1: Simple Explanation
# Goal: explain results in plain language for non-medical users

template_1_system = """You are a friendly health assistant. 
Explain medical results in simple, everyday language. 
Avoid technical terms. Keep responses short (3-5 sentences).
Be reassuring but honest."""

template_1_user = """A patient received their liver health screening result.

Prediction: {prediction}
Key values:
{features}

Explain this result in plain language, as if talking to a friend with no medical background.
Focus on the overall meaning, not the numbers."""

In [3]:
def simple_explanation(prediction, features):
    """Generate a simple explanation using Template 1."""
    
    user_message = template_1_user.format(
        prediction=prediction,
        features=features
    )
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_1_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.7,
        max_tokens=300
    )
    
    return response.choices[0].message.content

In [7]:
import pandas as pd

# Load raw dataset (easier for the AI to interpret)
data = pd.read_csv("Raw_Dataset/indian_liver_patient.csv")

# 3 diverse test cases
test_indices = [0, 315, 440] 

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]
    
    # Convert prediction: 1 = Liver Disease, 2 = No Liver Disease
    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"
    
    # Format features (drop the label so the AI doesn't see the answer)
    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])
    
    # Generate explanation
    print(f"Patient {i}: (row {idx}), Prediction: {prediction}")
    
    result = simple_explanation(prediction, features)
    print(result)

Patient 1: (row 0), Prediction: Liver Disease Detected


NameError: name 'simple_explanation' is not defined

### Template 2: Personalized Explanation

**Goal:** Provide a personalized explanation by linking the prediction to the patient’s specific values.

**Target user:** Patients who want a more relevant and tailored explanation of their results.

**Design choice:** Uses the patient’s actual data (e.g., age, lab values) to make the explanation more meaningful and engaging, while still keeping the language simple and easy to understand. Maintains a supportive tone without using complex medical terminology.

In [11]:
# Template 2: Personalized Explanation
# Goal: explain the result using the patient's specific values

template_2_system = """You are a personalized medical assistant.
Use the patient's specific data in your explanation.
Mention relevant patient values only if they help explain the prediction.
Explain what these values may indicate in simple language.
Keep explanations simple without unnecessary medical complexity.
Maintain a supportive and reassuring tone."""

template_2_user = """A patient received their liver health screening result.

Prediction: {prediction}
Key values:
{features}

Explain this result by directly referring to the patient’s specific values.
Highlight any notable indicators in a simple and personalized way."""

def personalized_explanation(prediction, features):
    """Generate a personalized explanation using Template 2."""
    
    user_message = template_2_user.format(
        prediction=prediction,
        features=features
    )
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_2_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.5,
        max_tokens=700
    )
    
    return response.choices[0].message.content

In [ ]:
import os

output_dir = "Generative_AI/example_outputs"
os.makedirs(output_dir, exist_ok=True)

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]

    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"

    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])

    result = personalized_explanation(prediction, features)

    file_path = f"{output_dir}/template_2_patient_{i}.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Patient {i}: (row {idx}), Prediction: {prediction}\n\n")
        f.write(result)

    print(f"Saved: {file_path}")


Patient 1: (row 0), Prediction: Liver Disease Detected
I'm here to help you understand your liver health screening result. Based on your test, it appears that liver disease has been detected. Let's break down what this means and look at your specific values.

First, your age, 65, is a factor to consider. As we age, our liver function can naturally decline, which may contribute to the development of liver disease.

Now, let's look at your bilirubin levels. Your Total Bilirubin is 0.7, and your Direct Bilirubin is 0.1. These values are within normal limits, which is a positive sign. Bilirubin is a waste product that your liver helps remove from your body. Normal levels suggest that your liver is functioning well in this regard.

However, your Alkaline Phosphatase (ALP) level is 187, which is slightly elevated. This enzyme is related to liver and bone health. An elevated ALP level can indicate liver damage or disease, as well as bone disorders. In your case, this might be a notable indic

### Template 4: Step-by-Step Guide

**Goal:** Walk through each lab value one by one to explain why the model made its prediction.

**Target user:** Patients who want to understand their result in detail, and doctors who need a transparent review of the reasoning.

**Design choice:** Step-by-step breakdown of each biomarker against its normal range, followed by a reasoning conclusion that ties everything together.

In [5]:
# Template 4: Step-by-Step Guide
# Goal: walk through each lab value and explain the reasoning behind the prediction

template_4_system = """You are a medical reasoning assistant.
When given lab values and a prediction, think step by step:
1. Check each value against its normal range.
2. Note which ones are too high, too low, or normal.
3. Explain how the abnormal values connect to the prediction.
4. End with a short conclusion that ties everything together.
Keep your language clear and use the reference ranges as part of your reasoning."""

template_4_user = """A machine learning model made the following prediction for a liver patient.

Prediction: {prediction}
Lab Values:
{features}

Go through each lab value step by step and explain whether it supports
or goes against the prediction. End with a clear reasoning conclusion."""


In [6]:
def step_by_step_guide(prediction, features):
    """Generate a step-by-step reasoning analysis using Template 4."""

    user_message = template_4_user.format(
        prediction=prediction,
        features=features
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_4_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.3,
        max_tokens=600
    )

    return response.choices[0].message.content


In [7]:
# Test Template 4 0n 3 test cases

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]

    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"

    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])

    print(f"Patient {i}: (row {idx}), Prediction: {prediction}")

    result = step_by_step_guide(prediction, features)
    print(result)


Patient 1: (row 0), Prediction: Liver Disease Detected
To evaluate the prediction of "Liver Disease Detected," let's examine each lab value against its normal range and see how it supports or contradicts the prediction.

1. **Age: 65** - Age itself is not a lab value but a demographic factor. However, it's known that the risk of liver disease can increase with age. Thus, being 65 might slightly increase the likelihood of liver disease, but it's not a direct indicator.

2. **Gender: Female** - Like age, gender is a demographic factor and not a lab value. Some liver diseases have different prevalence rates among genders, but without specific context, it's hard to draw a direct connection to the prediction.

3. **Total_Bilirubin: 0.7** - The normal range for total bilirubin is approximately 0.1 to 1.2 mg/dL. With a value of 0.7, this falls within the normal range. Normal bilirubin levels do not typically indicate liver disease, so this value does not support the prediction.

4. **Direct_B